# Scaffold vs MiniLM (one file)

Scaffold **generates** cross-domain scores from a closed theme lexicon + token overlap. Gold is evaluation-only.

No pip. No extra files. Works in Colab.


In [ ]:
#!/usr/bin/env python3
"""
Ontological scaffold vs MiniLM on the same 25-pair gold.

Scaffold predictions are GENERATED from concept labels + domain membership
+ a closed theme lexicon. Gold is used only at evaluation.
"""
from __future__ import annotations

import csv
from dataclasses import dataclass, asdict
from itertools import combinations
from pathlib import Path

OUT = Path.cwd() / "evaluation_output"

CONCEPTS = [
    ("pesticide residue contamination", "Agriculture"),
    ("chemical contaminant bioaccumulation", "Environment"),
    ("agricultural water use", "Agriculture"),
    ("groundwater aquifer depletion", "Environment"),
    ("pollinator habitat loss", "Agriculture"),
    ("biodiversity species extinction", "Environment"),
    ("soil carbon sequestration", "Agriculture"),
    ("atmospheric carbon dioxide concentration", "Environment"),
    ("agroforestry land use", "Agriculture"),
    ("deforestation tropical forest", "Environment"),
    ("aquaculture fish farming", "Agriculture"),
    ("antibiotic resistance surveillance", "Healthcare"),
    ("food supply chain logistics", "Agriculture"),
    ("pediatric nutrition intervention", "Healthcare"),
    ("plant disease surveillance", "Agriculture"),
    ("public health surveillance system", "Healthcare"),
    ("seed genetic modification", "Agriculture"),
    ("precision medicine genomic targeting", "Healthcare"),
    ("farm subsidy policy", "Agriculture"),
    ("income inequality Gini coefficient", "Socioeconomics"),
    ("crop yield variability", "Agriculture"),
    ("food insecurity household level", "Socioeconomics"),
    ("agricultural labor migration", "Agriculture"),
    ("rural urban economic divide", "Socioeconomics"),
    ("rural smallholder income", "Agriculture"),
    ("poverty alleviation microfinance", "Socioeconomics"),
    ("air quality particulate matter", "Environment"),
    ("chronic disease self-management", "Healthcare"),
    ("urban heat island effect", "Environment"),
    ("emergency department crowding", "Healthcare"),
    ("drought frequency projection", "Environment"),
    ("maternal mortality risk factor", "Healthcare"),
    ("freshwater availability stress", "Environment"),
    ("infectious disease contact tracing", "Healthcare"),
    ("carbon emission trading", "Environment"),
    ("carbon tax distributional effect", "Socioeconomics"),
    ("renewable energy transition", "Environment"),
    ("labor market automation displacement", "Socioeconomics"),
    ("ecosystem restoration practice", "Environment"),
    ("climate adaptation funding equity", "Socioeconomics"),
    ("sea level rise projection", "Environment"),
    ("disaster recovery economic resilience", "Socioeconomics"),
    ("social determinants of health", "Healthcare"),
    ("neighborhood segregation index", "Socioeconomics"),
    ("health insurance coverage gap", "Healthcare"),
    ("health expenditure household burden", "Socioeconomics"),
    ("mental health service access", "Healthcare"),
    ("unemployment benefit duration", "Socioeconomics"),
    ("health equity racial disparity", "Healthcare"),
    ("racial discrimination employment", "Socioeconomics"),
]
DOMAIN = {n: d for n, d in CONCEPTS}
NAMES = [n for n, _ in CONCEPTS]

GOLD_LIST = [
    ("pesticide residue contamination", "chemical contaminant bioaccumulation"),
    ("agricultural water use", "groundwater aquifer depletion"),
    ("pollinator habitat loss", "biodiversity species extinction"),
    ("soil carbon sequestration", "atmospheric carbon dioxide concentration"),
    ("agroforestry land use", "deforestation tropical forest"),
    ("aquaculture fish farming", "antibiotic resistance surveillance"),
    ("food supply chain logistics", "pediatric nutrition intervention"),
    ("plant disease surveillance", "public health surveillance system"),
    ("seed genetic modification", "precision medicine genomic targeting"),
    ("farm subsidy policy", "income inequality Gini coefficient"),
    ("crop yield variability", "food insecurity household level"),
    ("agricultural labor migration", "rural urban economic divide"),
    ("rural smallholder income", "poverty alleviation microfinance"),
    ("air quality particulate matter", "chronic disease self-management"),
    ("urban heat island effect", "emergency department crowding"),
    ("drought frequency projection", "maternal mortality risk factor"),
    ("freshwater availability stress", "infectious disease contact tracing"),
    ("carbon emission trading", "carbon tax distributional effect"),
    ("renewable energy transition", "labor market automation displacement"),
    ("ecosystem restoration practice", "climate adaptation funding equity"),
    ("sea level rise projection", "disaster recovery economic resilience"),
    ("social determinants of health", "neighborhood segregation index"),
    ("health insurance coverage gap", "health expenditure household burden"),
    ("mental health service access", "unemployment benefit duration"),
    ("health equity racial disparity", "racial discrimination employment"),
]
GOLD = {tuple(sorted(p)) for p in GOLD_LIST}
assert len(GOLD) == 25
assert all(DOMAIN[a] != DOMAIN[b] for a, b in GOLD_LIST)

# Closed theme lexicon. Assigned from the concept string only.
THEME_LEX = {
    "contaminant": {"pesticide", "residue", "contamination", "chemical", "contaminant", "bioaccumulation", "particulate", "air"},
    "water": {"water", "groundwater", "aquifer", "freshwater", "irrigation", "drought", "aquaculture"},
    "biodiversity": {"pollinator", "habitat", "biodiversity", "species", "extinction", "ecosystem"},
    "carbon": {"carbon", "sequestration", "atmospheric", "dioxide", "emission", "trading"},
    "land_forest": {"agroforestry", "land", "deforestation", "tropical", "forest", "soil", "restoration"},
    "amr_onehealth": {"aquaculture", "antibiotic", "resistance", "surveillance", "disease", "infectious", "public", "health", "plant"},
    "food_nutrition": {"food", "supply", "nutrition", "pediatric", "yield", "insecurity", "crop"},
    "genomics": {"seed", "genetic", "modification", "precision", "medicine", "genomic"},
    "income": {"farm", "subsidy", "income", "inequality", "gini", "poverty", "microfinance", "smallholder", "expenditure"},
    "labor": {"labor", "migration", "unemployment", "automation", "employment", "displacement", "market"},
    "rural_urban": {"rural", "urban", "divide", "neighborhood", "segregation"},
    "heat_emergency": {"heat", "island", "emergency", "department", "crowding"},
    "climate_disaster": {"drought", "sea", "level", "disaster", "recovery", "resilience", "climate", "adaptation"},
    "health_access": {"insurance", "coverage", "mental", "health", "equity", "racial", "disparity", "determinants", "mortality", "chronic", "disease"},
    "race": {"racial", "discrimination", "equity", "segregation", "neighborhood"},
    "energy": {"renewable", "energy", "transition", "automation", "labor"},
}
STOP = {"of", "the", "and", "a", "an", "to", "for", "in", "on", "level", "system", "practice", "effect", "use", "gap"}


def tokens(name: str) -> set:
    return {t for t in name.lower().replace("-", " ").split() if t not in STOP and len(t) > 2}


def themes(name: str) -> set:
    tok = tokens(name)
    return {th for th, lex in THEME_LEX.items() if tok & lex}


def scaffold_score(a: str, b: str) -> float:
    ta, tb = themes(a), themes(b)
    ka, kb = tokens(a), tokens(b)
    j_th = len(ta & tb) / len(ta | tb) if (ta | tb) else 0.0
    j_tok = len(ka & kb) / len(ka | kb) if (ka | kb) else 0.0
    conf = 0.65 * j_th + 0.35 * j_tok
    if not (ta & tb) and not (ka & kb):
        conf *= 0.15
    return conf


@dataclass
class Metrics:
    tp: int
    fp: int
    fn: int
    precision: float
    recall: float
    f1: float
    fdr: float
    average_precision: float = 0.0

    def to_dict(self):
        return asdict(self)


def nset(pairs):
    return {tuple(sorted(p)) for p in pairs}


def metrics(pred, gold, ap=0.0):
    p, g = nset(pred), nset(gold)
    tp, fp, fn = len(p & g), len(p - g), len(g - p)
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
    return Metrics(tp, fp, fn, prec, rec, f1, fp / (tp + fp) if (tp + fp) else 0.0, ap)


def ap_score(scored_pairs, gold):
    g = nset(gold)
    ordered = sorted(scored_pairs, key=lambda r: r[2], reverse=True)
    if not ordered or not g:
        return 0.0
    tp_so_far, s, seen = 0, 0.0, set()
    for rank, (a, b, _) in enumerate(ordered, 1):
        pair = tuple(sorted((a, b)))
        if pair in seen:
            continue
        seen.add(pair)
        if pair in g:
            tp_so_far += 1
            s += tp_so_far / rank
    return s / len(g)


def evaluate_threshold(scored, tau, gold):
    pred = [(a, b) for a, b, c in scored if c >= tau]
    return metrics(pred, gold)


def print_sweep(title, scored, gold, taus):
    print(f"\n{title}")
    print(f"{'τ':>6} {'Pred':>8} {'TP':>4} {'FP':>4} {'FN':>4} {'P':>8} {'R':>8} {'F1':>8} {'FDR':>8}")
    best = None
    for t in taus:
        m = evaluate_threshold(scored, t, gold)
        n = sum(1 for *_, c in scored if c >= t)
        print(f"{t:6.2f} {n:8d} {m.tp:4d} {m.fp:4d} {m.fn:4d} {m.precision:8.3f} {m.recall:8.3f} {m.f1:8.3f} {m.fdr:8.3f}")
        if best is None or m.f1 > best[1].f1:
            best = (t, m, n)
    return best


# EMBEDDING_PAIRS injected below

EMBEDDING_PAIRS = [
    ('rural urban economic divide', 'rural smallholder income', 0.6737722158432007),
    ('carbon emission trading', 'carbon tax distributional effect', 0.6249188184738159),
    ('agricultural labor migration', 'labor market automation displacement', 0.5696521997451782),
    ('pesticide residue contamination', 'chemical contaminant bioaccumulation', 0.5696088075637817),
    ('pollinator habitat loss', 'biodiversity species extinction', 0.5492212176322937),
    ('soil carbon sequestration', 'carbon emission trading', 0.5029525756835938),
    ('social determinants of health', 'health expenditure household burden', 0.491117388010025),
    ('soil carbon sequestration', 'carbon tax distributional effect', 0.46633180975914),
    ('pollinator habitat loss', 'ecosystem restoration practice', 0.46510133147239685),
    ('health equity racial disparity', 'racial discrimination employment', 0.4608444273471832),
    ('rural smallholder income', 'poverty alleviation microfinance', 0.44206732511520386),
    ('soil carbon sequestration', 'atmospheric carbon dioxide concentration', 0.43912389874458313),
    ('agricultural water use', 'freshwater availability stress', 0.43850183486938477),
    ('plant disease surveillance', 'public health surveillance system', 0.4265238642692566),
    ('atmospheric carbon dioxide concentration', 'carbon tax distributional effect', 0.4230305254459381),
    ('aquaculture fish farming', 'freshwater availability stress', 0.4197797477245331),
    ('health insurance coverage gap', 'health expenditure household burden', 0.41098493337631226),
    ('agroforestry land use', 'ecosystem restoration practice', 0.4007580876350403),
    ('soil carbon sequestration', 'ecosystem restoration practice', 0.40056124329566956),
    ('maternal mortality risk factor', 'health expenditure household burden', 0.39548546075820923),
    ('crop yield variability', 'drought frequency projection', 0.38712841272354126),
    ('agricultural water use', 'drought frequency projection', 0.38524702191352844),
    ('health expenditure household burden', 'health equity racial disparity', 0.38199880719184875),
    ('ecosystem restoration practice', 'disaster recovery economic resilience', 0.3805614411830902),
    ('groundwater aquifer depletion', 'soil carbon sequestration', 0.3772750496864319),
    ('ecosystem restoration practice', 'climate adaptation funding equity', 0.375776082277298),
    ('plant disease surveillance', 'infectious disease contact tracing', 0.37492117285728455),
    ('agroforestry land use', 'deforestation tropical forest', 0.37069427967071533),
    ('agricultural labor migration', 'rural urban economic divide', 0.36630839109420776),
    ('chemical contaminant bioaccumulation', 'soil carbon sequestration', 0.35892733931541443),
    ('food insecurity household level', 'social determinants of health', 0.35622504353523254),
    ('public health surveillance system', 'health expenditure household burden', 0.3516170382499695),
    ('agricultural water use', 'groundwater aquifer depletion', 0.34906014800071716),
    ('neighborhood segregation index', 'health equity racial disparity', 0.34788477420806885),
    ('farm subsidy policy', 'rural urban economic divide', 0.341370552778244),
    ('pediatric nutrition intervention', 'food insecurity household level', 0.3371061086654663),
    ('carbon emission trading', 'climate adaptation funding equity', 0.33554646372795105),
    ('pollinator habitat loss', 'freshwater availability stress', 0.33389154076576233),
    ('carbon tax distributional effect', 'renewable energy transition', 0.3291029930114746),
    ('food supply chain logistics', 'food insecurity household level', 0.3228117525577545),
    ('drought frequency projection', 'climate adaptation funding equity', 0.32269251346588135),
    ('farm subsidy policy', 'climate adaptation funding equity', 0.3219575583934784),
    ('antibiotic resistance surveillance', 'plant disease surveillance', 0.32130199670791626),
    ('deforestation tropical forest', 'plant disease surveillance', 0.3110485374927521),
    ('farm subsidy policy', 'poverty alleviation microfinance', 0.3095594644546509),
    ('agricultural water use', 'ecosystem restoration practice', 0.30931583046913147),
    ('seed genetic modification', 'precision medicine genomic targeting', 0.30148500204086304),
    ('farm subsidy policy', 'food insecurity household level', 0.29770493507385254),
    ('income inequality Gini coefficient', 'rural smallholder income', 0.2973008453845978),
    ('deforestation tropical forest', 'carbon tax distributional effect', 0.29713737964630127),
    ('pesticide residue contamination', 'air quality particulate matter', 0.2948984205722809),
    ('agricultural labor migration', 'racial discrimination employment', 0.29365232586860657),
    ('soil carbon sequestration', 'deforestation tropical forest', 0.29108381271362305),
    ('rural smallholder income', 'disaster recovery economic resilience', 0.2905317544937134),
    ('rural smallholder income', 'climate adaptation funding equity', 0.28993818163871765),
    ('emergency department crowding', 'disaster recovery economic resilience', 0.2888686954975128),
    ('agricultural labor migration', 'poverty alleviation microfinance', 0.2885318696498871),
    ('urban heat island effect', 'climate adaptation funding equity', 0.28784090280532837),
    ('freshwater availability stress', 'climate adaptation funding equity', 0.28707951307296753),
    ('rural smallholder income', 'health expenditure household burden', 0.28685125708580017),
    ('food insecurity household level', 'health equity racial disparity', 0.28125473856925964),
    ('agroforestry land use', 'climate adaptation funding equity', 0.27626466751098633),
    ('health insurance coverage gap', 'unemployment benefit duration', 0.2758421003818512),
    ('biodiversity species extinction', 'plant disease surveillance', 0.2745211720466614),
    ('biodiversity species extinction', 'soil carbon sequestration', 0.2729848027229309),
    ('income inequality Gini coefficient', 'maternal mortality risk factor', 0.27024585008621216),
    ('pesticide residue contamination', 'infectious disease contact tracing', 0.2698023319244385),
    ('aquaculture fish farming', 'ecosystem restoration practice', 0.269526869058609),
    ('soil carbon sequestration', 'climate adaptation funding equity', 0.2689253091812134),
    ('pollinator habitat loss', 'climate adaptation funding equity', 0.2678859531879425),
    ('agroforestry land use', 'poverty alleviation microfinance', 0.2656969130039215),
    ('plant disease surveillance', 'drought frequency projection', 0.26517152786254883),
    ('food insecurity household level', 'rural smallholder income', 0.2602683901786804),
    ('crop yield variability', 'food insecurity household level', 0.2573474049568176),
    ('pollinator habitat loss', 'deforestation tropical forest', 0.25676125288009644),
    ('freshwater availability stress', 'disaster recovery economic resilience', 0.255808562040329),
    ('biodiversity species extinction', 'crop yield variability', 0.2556959390640259),
    ('renewable energy transition', 'climate adaptation funding equity', 0.2555629312992096),
    ('income inequality Gini coefficient', 'health equity racial disparity', 0.2549895942211151),
    ('agricultural water use', 'rural urban economic divide', 0.2548840641975403),
    ('farm subsidy policy', 'carbon tax distributional effect', 0.2538844645023346),
    ('biodiversity species extinction', 'agroforestry land use', 0.250973641872406),
    ('farm subsidy policy', 'health expenditure household burden', 0.2505052089691162),
    ('pollinator habitat loss', 'atmospheric carbon dioxide concentration', 0.24946993589401245),
    ('air quality particulate matter', 'maternal mortality risk factor', 0.24667951464653015),
    ('chronic disease self-management', 'health expenditure household burden', 0.24549813568592072),
    ('deforestation tropical forest', 'climate adaptation funding equity', 0.24530096352100372),
    ('rural urban economic divide', 'urban heat island effect', 0.24510110914707184),
    ('urban heat island effect', 'carbon tax distributional effect', 0.24299165606498718),
    ('farm subsidy policy', 'health insurance coverage gap', 0.2428772747516632),
    ('food insecurity household level', 'agricultural labor migration', 0.24269039928913116),
    ('food supply chain logistics', 'pediatric nutrition intervention', 0.24235843122005463),
    ('air quality particulate matter', 'health expenditure household burden', 0.24142028391361237),
    ('agroforestry land use', 'food insecurity household level', 0.23992903530597687),
    ('income inequality Gini coefficient', 'social determinants of health', 0.23945151269435883),
    ('farm subsidy policy', 'racial discrimination employment', 0.23809435963630676),
    ('climate adaptation funding equity', 'health equity racial disparity', 0.23622675240039825),
    ('food insecurity household level', 'maternal mortality risk factor', 0.23612742125988007),
    ('mental health service access', 'racial discrimination employment', 0.23531576991081238),
    ('agroforestry land use', 'rural urban economic divide', 0.23496265709400177),
    ('agricultural water use', 'climate adaptation funding equity', 0.23489370942115784),
    ('crop yield variability', 'freshwater availability stress', 0.2334640920162201),
    ('chemical contaminant bioaccumulation', 'agricultural water use', 0.2316344827413559),
    ('atmospheric carbon dioxide concentration', 'climate adaptation funding equity', 0.22979046404361725),
    ('agricultural water use', 'food insecurity household level', 0.22770622372627258),
    ('biodiversity species extinction', 'disaster recovery economic resilience', 0.22552727162837982),
    ('agricultural water use', 'health expenditure household burden', 0.22548840939998627),
    ('chemical contaminant bioaccumulation', 'infectious disease contact tracing', 0.22420106828212738),
    ('agricultural labor migration', 'carbon emission trading', 0.22296319901943207),
    ('groundwater aquifer depletion', 'disaster recovery economic resilience', 0.22245709598064423),
    ('agricultural water use', 'deforestation tropical forest', 0.22230488061904907),
    ('farm subsidy policy', 'ecosystem restoration practice', 0.22185182571411133),
    ('plant disease surveillance', 'chronic disease self-management', 0.22069242596626282),
    ('air quality particulate matter', 'carbon tax distributional effect', 0.21989643573760986),
    ('crop yield variability', 'rural urban economic divide', 0.21833045780658722),
    ('soil carbon sequestration', 'air quality particulate matter', 0.2176925092935562),
    ('air quality particulate matter', 'infectious disease contact tracing', 0.21621409058570862),
    ('crop yield variability', 'carbon tax distributional effect', 0.2158064842224121),
    ('health expenditure household burden', 'mental health service access', 0.2143908441066742),
    ('rural smallholder income', 'labor market automation displacement', 0.2124694585800171),
    ('pesticide residue contamination', 'groundwater aquifer depletion', 0.21170677244663239),
    ('aquaculture fish farming', 'poverty alleviation microfinance', 0.21001394093036652),
    ('crop yield variability', 'climate adaptation funding equity', 0.2094815969467163),
    ('deforestation tropical forest', 'crop yield variability', 0.2088927924633026),
    ('groundwater aquifer depletion', 'pollinator habitat loss', 0.20843593776226044),
    ('pediatric nutrition intervention', 'farm subsidy policy', 0.20822320878505707),
    ('urban heat island effect', 'neighborhood segregation index', 0.20721404254436493),
    ('social determinants of health', 'neighborhood segregation index', 0.2066011130809784),
    ('urban heat island effect', 'disaster recovery economic resilience', 0.20574888586997986),
    ('chemical contaminant bioaccumulation', 'antibiotic resistance surveillance', 0.20457226037979126),
    ('poverty alleviation microfinance', 'health equity racial disparity', 0.204552561044693),
    ('farm subsidy policy', 'labor market automation displacement', 0.20420251786708832),
    ('rural urban economic divide', 'social determinants of health', 0.2039775550365448),
    ('infectious disease contact tracing', 'health expenditure household burden', 0.2038268893957138),
    ('rural urban economic divide', 'carbon emission trading', 0.20376767218112946),
    ('rural smallholder income', 'carbon tax distributional effect', 0.20352201163768768),
    ('agroforestry land use', 'disaster recovery economic resilience', 0.20245017111301422),
    ('public health surveillance system', 'food insecurity household level', 0.20105265080928802),
    ('rural urban economic divide', 'health equity racial disparity', 0.20025838911533356),
    ('rural smallholder income', 'ecosystem restoration practice', 0.19932149350643158),
    ('farm subsidy policy', 'renewable energy transition', 0.1992247849702835),
    ('pediatric nutrition intervention', 'rural smallholder income', 0.19868899881839752),
    ('food supply chain logistics', 'labor market automation displacement', 0.1982746571302414),
    ('rural smallholder income', 'carbon emission trading', 0.19724823534488678),
    ('chemical contaminant bioaccumulation', 'agroforestry land use', 0.19662761688232422),
    ('pesticide residue contamination', 'food insecurity household level', 0.19627349078655243),
    ('biodiversity species extinction', 'climate adaptation funding equity', 0.19619302451610565),
    ('renewable energy transition', 'disaster recovery economic resilience', 0.195293590426445),
    ('pesticide residue contamination', 'biodiversity species extinction', 0.19455455243587494),
    ('food supply chain logistics', 'carbon emission trading', 0.19455182552337646),
    ('farm subsidy policy', 'income inequality Gini coefficient', 0.1944337636232376),
    ('deforestation tropical forest', 'disaster recovery economic resilience', 0.19428002834320068),
    ('groundwater aquifer depletion', 'agroforestry land use', 0.19362890720367432),
    ('aquaculture fish farming', 'food insecurity household level', 0.19348984956741333),
    ('pediatric nutrition intervention', 'health expenditure household burden', 0.1920243203639984),
    ('carbon emission trading', 'labor market automation displacement', 0.19105184078216553),
    ('farm subsidy policy', 'carbon emission trading', 0.1891041100025177),
    ('poverty alleviation microfinance', 'ecosystem restoration practice', 0.18842251598834991),
    ('agricultural labor migration', 'renewable energy transition', 0.18840067088603973),
    ('public health surveillance system', 'air quality particulate matter', 0.188169464468956),
    ('agroforestry land use', 'emergency department crowding', 0.1870151311159134),
    ('agroforestry land use', 'drought frequency projection', 0.18507273495197296),
    ('chemical contaminant bioaccumulation', 'aquaculture fish farming', 0.18437910079956055),
    ('pollinator habitat loss', 'maternal mortality risk factor', 0.18332314491271973),
    ('pesticide residue contamination', 'antibiotic resistance surveillance', 0.1833043247461319),
    ('agricultural labor migration', 'unemployment benefit duration', 0.18299154937267303),
    ('pediatric nutrition intervention', 'poverty alleviation microfinance', 0.18254344165325165),
    ('drought frequency projection', 'disaster recovery economic resilience', 0.18225151300430298),
    ('deforestation tropical forest', 'agricultural labor migration', 0.18218326568603516),
    ('deforestation tropical forest', 'rural smallholder income', 0.18177756667137146),
    ('deforestation tropical forest', 'poverty alleviation microfinance', 0.18076147139072418),
    ('agroforestry land use', 'carbon emission trading', 0.18072542548179626),
    ('emergency department crowding', 'neighborhood segregation index', 0.18051820993423462),
    ('agricultural labor migration', 'health expenditure household burden', 0.17978812754154205),
    ('pollinator habitat loss', 'air quality particulate matter', 0.1796371042728424),
    ('climate adaptation funding equity', 'health insurance coverage gap', 0.17947721481323242),
    ('food insecurity household level', 'freshwater availability stress', 0.17923228442668915),
    ('urban heat island effect', 'emergency department crowding', 0.17803017795085907),
    ('income inequality Gini coefficient', 'crop yield variability', 0.17775565385818481),
    ('pollinator habitat loss', 'neighborhood segregation index', 0.1763731688261032),
    ('pesticide residue contamination', 'freshwater availability stress', 0.17575299739837646),
    ('biodiversity species extinction', 'seed genetic modification', 0.1757003217935562),
    ('aquaculture fish farming', 'renewable energy transition', 0.1747046709060669),
    ('labor market automation displacement', 'health insurance coverage gap', 0.17467518150806427),
    ('pollinator habitat loss', 'disaster recovery economic resilience', 0.17422226071357727),
    ('rural urban economic divide', 'health insurance coverage gap', 0.17385125160217285),
    ('agroforestry land use', 'freshwater availability stress', 0.17351199686527252),
    ('income inequality Gini coefficient', 'health insurance coverage gap', 0.1729898452758789),
    ('air quality particulate matter', 'social determinants of health', 0.17123529314994812),
    ('agricultural water use', 'carbon emission trading', 0.17100928723812103),
    ('pesticide residue contamination', 'maternal mortality risk factor', 0.17052507400512695),
    ('pollinator habitat loss', 'carbon tax distributional effect', 0.1690259724855423),
    ('biodiversity species extinction', 'aquaculture fish farming', 0.16865719854831696),
    ('biodiversity species extinction', 'carbon tax distributional effect', 0.16851906478405),
    ('agricultural labor migration', 'disaster recovery economic resilience', 0.16758617758750916),
    ('agricultural labor migration', 'infectious disease contact tracing', 0.16612094640731812),
    ('seed genetic modification', 'climate adaptation funding equity', 0.16448433697223663),
    ('agricultural water use', 'poverty alleviation microfinance', 0.16442689299583435),
    ('aquaculture fish farming', 'racial discrimination employment', 0.16270935535430908),
    ('income inequality Gini coefficient', 'agricultural labor migration', 0.16227149963378906),
    ('plant disease surveillance', 'social determinants of health', 0.1620045155286789),
    ('soil carbon sequestration', 'drought frequency projection', 0.16195820271968842),
    ('chronic disease self-management', 'ecosystem restoration practice', 0.16138170659542084),
    ('pollinator habitat loss', 'drought frequency projection', 0.16083599627017975),
    ('rural urban economic divide', 'maternal mortality risk factor', 0.16013017296791077),
    ('pollinator habitat loss', 'carbon emission trading', 0.15835222601890564),
    ('carbon tax distributional effect', 'ecosystem restoration practice', 0.1579262912273407),
    ('poverty alleviation microfinance', 'freshwater availability stress', 0.15654578804969788),
    ('biodiversity species extinction', 'neighborhood segregation index', 0.1563848853111267),
    ('food supply chain logistics', 'rural urban economic divide', 0.1559540182352066),
    ('pesticide residue contamination', 'atmospheric carbon dioxide concentration', 0.15538224577903748),
    ('farm subsidy policy', 'health equity racial disparity', 0.15498919785022736),
    ('emergency department crowding', 'labor market automation displacement', 0.15497912466526031),
    ('crop yield variability', 'renewable energy transition', 0.15473328530788422),
    ('groundwater aquifer depletion', 'climate adaptation funding equity', 0.1536426842212677),
    ('soil carbon sequestration', 'freshwater availability stress', 0.15326102077960968),
    ('rural smallholder income', 'neighborhood segregation index', 0.1522131711244583),
    ('pesticide residue contamination', 'ecosystem restoration practice', 0.15107642114162445),
    ('public health surveillance system', 'neighborhood segregation index', 0.15094712376594543),
    ('agricultural labor migration', 'climate adaptation funding equity', 0.1507626622915268),
    ('aquaculture fish farming', 'climate adaptation funding equity', 0.15054349601268768),
    ('poverty alleviation microfinance', 'carbon emission trading', 0.1499304622411728),
    ('carbon tax distributional effect', 'social determinants of health', 0.14868752658367157),
    ('drought frequency projection', 'carbon tax distributional effect', 0.14824199676513672),
    ('air quality particulate matter', 'neighborhood segregation index', 0.14794738590717316),
    ('rural smallholder income', 'maternal mortality risk factor', 0.14740893244743347),
    ('agricultural water use', 'air quality particulate matter', 0.14711908996105194),
    ('plant disease surveillance', 'ecosystem restoration practice', 0.14709359407424927),
    ('agricultural labor migration', 'neighborhood segregation index', 0.14706693589687347),
    ('chemical contaminant bioaccumulation', 'seed genetic modification', 0.14653359353542328),
    ('air quality particulate matter', 'climate adaptation funding equity', 0.14586728811264038),
    ('rural smallholder income', 'freshwater availability stress', 0.1458064317703247),
    ('aquaculture fish farming', 'drought frequency projection', 0.1456182450056076),
    ('farm subsidy policy', 'drought frequency projection', 0.14543864130973816),
    ('agroforestry land use', 'carbon tax distributional effect', 0.14536936581134796),
    ('food insecurity household level', 'mental health service access', 0.14399103820323944),
    ('poverty alleviation microfinance', 'social determinants of health', 0.14362819492816925),
    ('pollinator habitat loss', 'urban heat island effect', 0.14307335019111633),
    ('agroforestry land use', 'income inequality Gini coefficient', 0.14301981031894684),
    ('food supply chain logistics', 'disaster recovery economic resilience', 0.14294777810573578),
    ('food supply chain logistics', 'emergency department crowding', 0.14242786169052124),
    ('chemical contaminant bioaccumulation', 'plant disease surveillance', 0.1420821249485016),
    ('crop yield variability', 'disaster recovery economic resilience', 0.14162173867225647),
    ('rural urban economic divide', 'emergency department crowding', 0.1415625810623169),
    ('pollinator habitat loss', 'poverty alleviation microfinance', 0.14132757484912872),
    ('food supply chain logistics', 'ecosystem restoration practice', 0.1407877504825592),
    ('neighborhood segregation index', 'mental health service access', 0.14005930721759796),
    ('renewable energy transition', 'health insurance coverage gap', 0.13939617574214935),
    ('renewable energy transition', 'labor market automation displacement', 0.1389538198709488),
    ('poverty alleviation microfinance', 'emergency department crowding', 0.1388012021780014),
    ('disaster recovery economic resilience', 'social determinants of health', 0.1385401040315628),
    ('maternal mortality risk factor', 'climate adaptation funding equity', 0.1383436918258667),
    ('food supply chain logistics', 'poverty alleviation microfinance', 0.13820981979370117),
    ('income inequality Gini coefficient', 'drought frequency projection', 0.138131245970726),
    ('neighborhood segregation index', 'health insurance coverage gap', 0.13811853528022766),
    ('biodiversity species extinction', 'poverty alleviation microfinance', 0.1373746544122696),
    ('pollinator habitat loss', 'food insecurity household level', 0.13691508769989014),
    ('atmospheric carbon dioxide concentration', 'crop yield variability', 0.1365911066532135),
    ('carbon emission trading', 'disaster recovery economic resilience', 0.13621661067008972),
    ('soil carbon sequestration', 'renewable energy transition', 0.13557352125644684),
    ('pollinator habitat loss', 'renewable energy transition', 0.13515686988830566),
    ('crop yield variability', 'unemployment benefit duration', 0.1342279016971588),
    ('agricultural labor migration', 'freshwater availability stress', 0.1336880326271057),
    ('aquaculture fish farming', 'sea level rise projection', 0.13319538533687592),
    ('farm subsidy policy', 'freshwater availability stress', 0.13275139033794403),
    ('food insecurity household level', 'health insurance coverage gap', 0.13245005905628204),
    ('farm subsidy policy', 'disaster recovery economic resilience', 0.1314997375011444),
    ('chemical contaminant bioaccumulation', 'pollinator habitat loss', 0.1313857138156891),
    ('rural urban economic divide', 'renewable energy transition', 0.13095995783805847),
    ('agricultural labor migration', 'ecosystem restoration practice', 0.13053205609321594),
    ('emergency department crowding', 'health expenditure household burden', 0.13049837946891785),
    ('rural smallholder income', 'health equity racial disparity', 0.1300453096628189),
    ('air quality particulate matter', 'health equity racial disparity', 0.12920896708965302),
    ('plant disease surveillance', 'maternal mortality risk factor', 0.1274392157793045),
    ('soil carbon sequestration', 'rural urban economic divide', 0.12667036056518555),
    ('carbon emission trading', 'racial discrimination employment', 0.1262866109609604),
    ('rural smallholder income', 'racial discrimination employment', 0.12550003826618195),
    ('seed genetic modification', 'ecosystem restoration practice', 0.12521417438983917),
    ('food insecurity household level', 'emergency department crowding', 0.12438933551311493),
    ('deforestation tropical forest', 'infectious disease contact tracing', 0.12431281059980392),
    ('crop yield variability', 'carbon emission trading', 0.12408196926116943),
    ('farm subsidy policy', 'maternal mortality risk factor', 0.12402938306331635),
    ('agricultural labor migration', 'emergency department crowding', 0.12358170747756958),
    ('drought frequency projection', 'unemployment benefit duration', 0.12332617491483688),
    ('deforestation tropical forest', 'income inequality Gini coefficient', 0.12326739728450775),
    ('plant disease surveillance', 'food insecurity household level', 0.12316625565290451),
    ('aquaculture fish farming', 'rural urban economic divide', 0.1230846419930458),
    ('deforestation tropical forest', 'rural urban economic divide', 0.12305983155965805),
    ('agricultural water use', 'renewable energy transition', 0.12301632761955261),
    ('agricultural labor migration', 'health insurance coverage gap', 0.12294808030128479),
    ('soil carbon sequestration', 'labor market automation displacement', 0.12266942113637924),
    ('freshwater availability stress', 'health expenditure household burden', 0.12266187369823456),
    ('agroforestry land use', 'health expenditure household burden', 0.12261713296175003),
    ('urban heat island effect', 'health insurance coverage gap', 0.12219677120447159),
    ('pesticide residue contamination', 'rural urban economic divide', 0.12163961678743362),
    ('groundwater aquifer depletion', 'unemployment benefit duration', 0.12136615067720413),
    ('deforestation tropical forest', 'seed genetic modification', 0.1212131530046463),
    ('infectious disease contact tracing', 'neighborhood segregation index', 0.12097761780023575),
    ('chemical contaminant bioaccumulation', 'food insecurity household level', 0.1209733784198761),
    ('chemical contaminant bioaccumulation', 'carbon tax distributional effect', 0.120916947722435),
    ('plant disease surveillance', 'health expenditure household burden', 0.12089022248983383),
    ('plant disease surveillance', 'emergency department crowding', 0.1207796111702919),
    ('public health surveillance system', 'farm subsidy policy', 0.12054941803216934),
    ('emergency department crowding', 'ecosystem restoration practice', 0.11977951228618622),
    ('carbon tax distributional effect', 'sea level rise projection', 0.11961168050765991),
    ('pesticide residue contamination', 'health expenditure household burden', 0.11953695863485336),
    ('seed genetic modification', 'maternal mortality risk factor', 0.1194411888718605),
    ('pesticide residue contamination', 'public health surveillance system', 0.1193816140294075),
    ('pollinator habitat loss', 'emergency department crowding', 0.11889780312776566),
    ('infectious disease contact tracing', 'labor market automation displacement', 0.11868821829557419),
    ('rural smallholder income', 'emergency department crowding', 0.11861805617809296),
    ('chemical contaminant bioaccumulation', 'maternal mortality risk factor', 0.1183578297495842),
    ('agroforestry land use', 'maternal mortality risk factor', 0.11825478821992874),
    ('urban heat island effect', 'labor market automation displacement', 0.11823344975709915),
    ('crop yield variability', 'ecosystem restoration practice', 0.11812638491392136),
    ('agroforestry land use', 'renewable energy transition', 0.11786811053752899),
    ('ecosystem restoration practice', 'neighborhood segregation index', 0.11783704161643982),
    ('aquaculture fish farming', 'pediatric nutrition intervention', 0.11752152442932129),
    ('groundwater aquifer depletion', 'crop yield variability', 0.11718800663948059),
    ('pediatric nutrition intervention', 'climate adaptation funding equity', 0.117156982421875),
    ('farm subsidy policy', 'neighborhood segregation index', 0.11614197492599487),
    ('rural smallholder income', 'health insurance coverage gap', 0.11562404781579971),
    ('food insecurity household level', 'urban heat island effect', 0.11551393568515778),
    ('seed genetic modification', 'renewable energy transition', 0.11528538912534714),
    ('poverty alleviation microfinance', 'renewable energy transition', 0.11443470418453217),
    ('agricultural water use', 'emergency department crowding', 0.11337486654520035),
    ('aquaculture fish farming', 'carbon emission trading', 0.1125267744064331),
    ('soil carbon sequestration', 'poverty alleviation microfinance', 0.11243905127048492),
    ('agroforestry land use', 'urban heat island effect', 0.1118980124592781),
    ('agricultural labor migration', 'maternal mortality risk factor', 0.11176847666501999),
    ('plant disease surveillance', 'air quality particulate matter', 0.11138138920068741),
    ('air quality particulate matter', 'emergency department crowding', 0.11122720688581467),
    ('agroforestry land use', 'neighborhood segregation index', 0.11110629886388779),
    ('deforestation tropical forest', 'farm subsidy policy', 0.11105705052614212),
    ('biodiversity species extinction', 'antibiotic resistance surveillance', 0.11099286377429962),
    ('crop yield variability', 'maternal mortality risk factor', 0.11094905436038971),
    ('groundwater aquifer depletion', 'labor market automation displacement', 0.1107868179678917),
    ('income inequality Gini coefficient', 'mental health service access', 0.11065062880516052),
    ('food supply chain logistics', 'racial discrimination employment', 0.1100509986281395),
    ('agroforestry land use', 'labor market automation displacement', 0.1098780483007431),
    ('maternal mortality risk factor', 'disaster recovery economic resilience', 0.10970833152532578),
    ('renewable energy transition', 'unemployment benefit duration', 0.10968868434429169),
    ('biodiversity species extinction', 'maternal mortality risk factor', 0.10943257808685303),
    ('drought frequency projection', 'infectious disease contact tracing', 0.10933051258325577),
    ('groundwater aquifer depletion', 'rural smallholder income', 0.10896653681993484),
    ('agricultural labor migration', 'carbon tax distributional effect', 0.108929343521595),
    ('rural smallholder income', 'social determinants of health', 0.10817849636077881),
    ('pesticide residue contamination', 'carbon emission trading', 0.10816022753715515),
    ('poverty alleviation microfinance', 'health insurance coverage gap', 0.10812055319547653),
    ('deforestation tropical forest', 'chronic disease self-management', 0.1081024557352066),
    ('agricultural water use', 'biodiversity species extinction', 0.10800530016422272),
    ('antibiotic resistance surveillance', 'health expenditure household burden', 0.10723473131656647),
    ('aquaculture fish farming', 'antibiotic resistance surveillance', 0.10655730962753296),
    ('agroforestry land use', 'air quality particulate matter', 0.10654684901237488),
    ('labor market automation displacement', 'sea level rise projection', 0.10653474926948547),
    ('pediatric nutrition intervention', 'rural urban economic divide', 0.10648565739393234),
    ('soil carbon sequestration', 'disaster recovery economic resilience', 0.10596302896738052),
    ('pediatric nutrition intervention', 'ecosystem restoration practice', 0.10576377063989639),
    ('agroforestry land use', 'public health surveillance system', 0.10480798035860062),
    ('farm subsidy policy', 'unemployment benefit duration', 0.10466855019330978),
    ('atmospheric carbon dioxide concentration', 'public health surveillance system', 0.10439258068799973),
    ('atmospheric carbon dioxide concentration', 'health expenditure household burden', 0.10435310751199722),
    ('aquaculture fish farming', 'precision medicine genomic targeting', 0.10394959896802902),
    ('agricultural water use', 'disaster recovery economic resilience', 0.10390117764472961),
    ('agroforestry land use', 'racial discrimination employment', 0.10358002036809921),
    ('public health surveillance system', 'drought frequency projection', 0.10339158773422241),
    ('crop yield variability', 'poverty alleviation microfinance', 0.10306667536497116),
    ('seed genetic modification', 'poverty alleviation microfinance', 0.10299099236726761),
    ('pediatric nutrition intervention', 'crop yield variability', 0.10263875126838684),
    ('climate adaptation funding equity', 'sea level rise projection', 0.10211250185966492),
    ('soil carbon sequestration', 'sea level rise projection', 0.10202287137508392),
    ('pediatric nutrition intervention', 'seed genetic modification', 0.10194415599107742),
    ('public health surveillance system', 'disaster recovery economic resilience', 0.10174683481454849),
    ('agricultural water use', 'income inequality Gini coefficient', 0.10152485966682434),
    ('atmospheric carbon dioxide concentration', 'emergency department crowding', 0.10098054260015488),
    ('income inequality Gini coefficient', 'sea level rise projection', 0.10097568482160568),
    ('farm subsidy policy', 'social determinants of health', 0.10068569332361221),
    ('urban heat island effect', 'unemployment benefit duration', 0.10053186118602753),
    ('food supply chain logistics', 'renewable energy transition', 0.10012572258710861),
    ('crop yield variability', 'health expenditure household burden', 0.10005758702754974),
    ('food insecurity household level', 'carbon emission trading', 0.09976311773061752),
    ('rural smallholder income', 'renewable energy transition', 0.09961629658937454),
    ('aquaculture fish farming', 'health equity racial disparity', 0.0993427261710167),
    ('farm subsidy policy', 'emergency department crowding', 0.09884458780288696),
    ('biodiversity species extinction', 'agricultural labor migration', 0.09786652773618698),
    ('sea level rise projection', 'disaster recovery economic resilience', 0.09756612032651901),
    ('food supply chain logistics', 'public health surveillance system', 0.09741997718811035),
    ('pollinator habitat loss', 'rural urban economic divide', 0.09706524759531021),
    ('mental health service access', 'unemployment benefit duration', 0.09690435975790024),
    ('climate adaptation funding equity', 'social determinants of health', 0.09670158475637436),
    ('disaster recovery economic resilience', 'health insurance coverage gap', 0.09640620648860931),
    ('drought frequency projection', 'health expenditure household burden', 0.096029132604599),
    ('deforestation tropical forest', 'health expenditure household burden', 0.095903180539608),
    ('chemical contaminant bioaccumulation', 'crop yield variability', 0.0948929414153099),
    ('food supply chain logistics', 'health expenditure household burden', 0.09480801969766617),
    ('agroforestry land use', 'health equity racial disparity', 0.09469286352396011),
    ('agricultural water use', 'atmospheric carbon dioxide concentration', 0.09450249373912811),
    ('agricultural water use', 'maternal mortality risk factor', 0.09421831369400024),
    ('carbon tax distributional effect', 'health insurance coverage gap', 0.09383716434240341),
    ('urban heat island effect', 'social determinants of health', 0.09351295232772827),
    ('pollinator habitat loss', 'income inequality Gini coefficient', 0.09344847500324249),
    ('food insecurity household level', 'air quality particulate matter', 0.09303157031536102),
    ('agroforestry land use', 'pediatric nutrition intervention', 0.09290408343076706),
    ('agricultural labor migration', 'health equity racial disparity', 0.09271278977394104),
    ('rural smallholder income', 'urban heat island effect', 0.09268031269311905),
    ('pesticide residue contamination', 'carbon tax distributional effect', 0.09267271310091019),
    ('antibiotic resistance surveillance', 'seed genetic modification', 0.09265441447496414),
    ('agricultural water use', 'labor market automation displacement', 0.09252451360225677),
    ('food supply chain logistics', 'freshwater availability stress', 0.09182227402925491),
    ('plant disease surveillance', 'health insurance coverage gap', 0.09157112240791321),
    ('precision medicine genomic targeting', 'poverty alleviation microfinance', 0.09121917188167572),
    ('atmospheric carbon dioxide concentration', 'maternal mortality risk factor', 0.09091846644878387),
    ('chronic disease self-management', 'climate adaptation funding equity', 0.09074593335390091),
    ('emergency department crowding', 'climate adaptation funding equity', 0.09060433506965637),
    ('plant disease surveillance', 'precision medicine genomic targeting', 0.09032196551561356),
    ('social determinants of health', 'racial discrimination employment', 0.09028059244155884),
    ('climate adaptation funding equity', 'mental health service access', 0.08996593207120895),
    ('deforestation tropical forest', 'maternal mortality risk factor', 0.08987961709499359),
    ('agroforestry land use', 'sea level rise projection', 0.08953968435525894),
    ('renewable energy transition', 'health expenditure household burden', 0.0892120823264122),
    ('chemical contaminant bioaccumulation', 'disaster recovery economic resilience', 0.08914594352245331),
    ('biodiversity species extinction', 'racial discrimination employment', 0.08912748843431473),
    ('farm subsidy policy', 'mental health service access', 0.08876468986272812),
    ('agricultural water use', 'infectious disease contact tracing', 0.08860577642917633),
    ('agricultural labor migration', 'drought frequency projection', 0.08814945816993713),
    ('aquaculture fish farming', 'labor market automation displacement', 0.08792950958013535),
    ('maternal mortality risk factor', 'carbon tax distributional effect', 0.08734231442213058),
    ('groundwater aquifer depletion', 'emergency department crowding', 0.08709147572517395),
    ('emergency department crowding', 'freshwater availability stress', 0.08687791228294373),
    ('food supply chain logistics', 'chronic disease self-management', 0.08684297651052475),
    ('plant disease surveillance', 'freshwater availability stress', 0.08675626665353775),
    ('groundwater aquifer depletion', 'farm subsidy policy', 0.08597303926944733),
    ('freshwater availability stress', 'labor market automation displacement', 0.0857866033911705),
    ('freshwater availability stress', 'social determinants of health', 0.08513301610946655),
    ('pediatric nutrition intervention', 'freshwater availability stress', 0.0850638598203659),
    ('agroforestry land use', 'chronic disease self-management', 0.08504585921764374),
    ('agricultural water use', 'carbon tax distributional effect', 0.08419731259346008),
    ('aquaculture fish farming', 'health expenditure household burden', 0.08411196619272232),
    ('aquaculture fish farming', 'social determinants of health', 0.08321361243724823),
    ('agricultural labor migration', 'urban heat island effect', 0.08312374353408813),
    ('seed genetic modification', 'drought frequency projection', 0.08290928602218628),
    ('seed genetic modification', 'income inequality Gini coefficient', 0.0824427455663681),
    ('health insurance coverage gap', 'racial discrimination employment', 0.08214298635721207),
    ('air quality particulate matter', 'health insurance coverage gap', 0.08168963342905045),
    ('freshwater availability stress', 'health equity racial disparity', 0.08159281313419342),
    ('food supply chain logistics', 'mental health service access', 0.08155513554811478),
    ('maternal mortality risk factor', 'neighborhood segregation index', 0.0814884752035141),
    ('income inequality Gini coefficient', 'carbon emission trading', 0.08081106096506119),
    ('disaster recovery economic resilience', 'health equity racial disparity', 0.08081025630235672),
    ('deforestation tropical forest', 'public health surveillance system', 0.08065126836299896),
    ('agricultural labor migration', 'social determinants of health', 0.08063296973705292),
    ('urban heat island effect', 'racial discrimination employment', 0.0805242732167244),
    ('food insecurity household level', 'chronic disease self-management', 0.08001765608787537),
    ('agricultural water use', 'pediatric nutrition intervention', 0.0797910988330841),
    ('poverty alleviation microfinance', 'urban heat island effect', 0.07965824753046036),
    ('agroforestry land use', 'infectious disease contact tracing', 0.07954837381839752),
    ('sea level rise projection', 'mental health service access', 0.07953625172376633),
    ('biodiversity species extinction', 'infectious disease contact tracing', 0.07936210185289383),
    ('antibiotic resistance surveillance', 'food insecurity household level', 0.0793270543217659),
    ('farm subsidy policy', 'chronic disease self-management', 0.07909523695707321),
    ('deforestation tropical forest', 'emergency department crowding', 0.07831817120313644),
    ('maternal mortality risk factor', 'freshwater availability stress', 0.07827681303024292),
    ('poverty alleviation microfinance', 'chronic disease self-management', 0.07809724658727646),
    ('plant disease surveillance', 'mental health service access', 0.07769423723220825),
    ('drought frequency projection', 'mental health service access', 0.0768834799528122),
    ('food supply chain logistics', 'carbon tax distributional effect', 0.07674803584814072),
    ('antibiotic resistance surveillance', 'farm subsidy policy', 0.07647917419672012),
    ('pediatric nutrition intervention', 'plant disease surveillance', 0.0762014165520668),
    ('urban heat island effect', 'health expenditure household burden', 0.07605067640542984),
    ('groundwater aquifer depletion', 'rural urban economic divide', 0.07528789341449738),
    ('rural urban economic divide', 'ecosystem restoration practice', 0.075130894780159),
    ('agricultural water use', 'public health surveillance system', 0.0750621110200882),
    ('sea level rise projection', 'health expenditure household burden', 0.07473897188901901),
    ('plant disease surveillance', 'renewable energy transition', 0.07410233467817307),
    ('unemployment benefit duration', 'health equity racial disparity', 0.07377910614013672),
    ('public health surveillance system', 'urban heat island effect', 0.07361981272697449),
    ('poverty alleviation microfinance', 'maternal mortality risk factor', 0.07361749559640884),
    ('deforestation tropical forest', 'food supply chain logistics', 0.07306932657957077),
    ('emergency department crowding', 'racial discrimination employment', 0.07205543667078018),
    ('chronic disease self-management', 'drought frequency projection', 0.0719766765832901),
    ('food supply chain logistics', 'sea level rise projection', 0.07180847227573395),
    ('soil carbon sequestration', 'pediatric nutrition intervention', 0.07165118306875229),
    ('disaster recovery economic resilience', 'mental health service access', 0.07150428742170334),
    ('pollinator habitat loss', 'infectious disease contact tracing', 0.07118165493011475),
    ('aquaculture fish farming', 'public health surveillance system', 0.07113499194383621),
    ('groundwater aquifer depletion', 'poverty alleviation microfinance', 0.07067780941724777),
    ('groundwater aquifer depletion', 'maternal mortality risk factor', 0.07042208313941956),
    ('rural urban economic divide', 'air quality particulate matter', 0.07015898823738098),
    ('chemical contaminant bioaccumulation', 'agricultural labor migration', 0.06985986977815628),
    ('ecosystem restoration practice', 'health equity racial disparity', 0.06910251080989838),
    ('atmospheric carbon dioxide concentration', 'income inequality Gini coefficient', 0.06852760165929794),
    ('infectious disease contact tracing', 'sea level rise projection', 0.06835392117500305),
    ('chemical contaminant bioaccumulation', 'emergency department crowding', 0.06828378885984421),
    ('food supply chain logistics', 'drought frequency projection', 0.06776291877031326),
    ('chemical contaminant bioaccumulation', 'public health surveillance system', 0.06735065579414368),
    ('income inequality Gini coefficient', 'renewable energy transition', 0.06732448935508728),
    ('biodiversity species extinction', 'chronic disease self-management', 0.06722615659236908),
    ('atmospheric carbon dioxide concentration', 'chronic disease self-management', 0.06710570305585861),
    ('soil carbon sequestration', 'emergency department crowding', 0.06669089198112488),
    ('atmospheric carbon dioxide concentration', 'social determinants of health', 0.0666578933596611),
    ('sea level rise projection', 'unemployment benefit duration', 0.06639210879802704),
    ('rural smallholder income', 'drought frequency projection', 0.06637649238109589),
    ('poverty alleviation microfinance', 'drought frequency projection', 0.06608670949935913),
    ('poverty alleviation microfinance', 'mental health service access', 0.06606681644916534),
    ('agricultural water use', 'sea level rise projection', 0.06587164103984833),
    ('deforestation tropical forest', 'racial discrimination employment', 0.06563559174537659),
    ('deforestation tropical forest', 'aquaculture fish farming', 0.0652243122458458),
    ('crop yield variability', 'sea level rise projection', 0.06515700370073318),
    ('agroforestry land use', 'mental health service access', 0.06503737717866898),
    ('pediatric nutrition intervention', 'disaster recovery economic resilience', 0.06452787667512894),
    ('drought frequency projection', 'racial discrimination employment', 0.06430192291736603),
    ('urban heat island effect', 'health equity racial disparity', 0.0639026015996933),
    ('agricultural water use', 'mental health service access', 0.06387010216712952),
    ('rural smallholder income', 'mental health service access', 0.06377153098583221),
    ('food supply chain logistics', 'climate adaptation funding equity', 0.06366715580224991),
    ('sea level rise projection', 'racial discrimination employment', 0.06354580819606781),
    ('biodiversity species extinction', 'public health surveillance system', 0.06341525912284851),
    ('biodiversity species extinction', 'emergency department crowding', 0.06321554630994797),
    ('pollinator habitat loss', 'health expenditure household burden', 0.06296998262405396),
    ('maternal mortality risk factor', 'unemployment benefit duration', 0.06287577003240585),
    ('freshwater availability stress', 'unemployment benefit duration', 0.06150170415639877),
    ('labor market automation displacement', 'health equity racial disparity', 0.061470553278923035),
    ('pollinator habitat loss', 'labor market automation displacement', 0.06119336187839508),
    ('chronic disease self-management', 'disaster recovery economic resilience', 0.06110270693898201),
    ('chemical contaminant bioaccumulation', 'food supply chain logistics', 0.06033724918961525),
    ('aquaculture fish farming', 'chronic disease self-management', 0.05993819981813431),
    ('ecosystem restoration practice', 'racial discrimination employment', 0.059866763651371),
    ('drought frequency projection', 'labor market automation displacement', 0.059668395668268204),
    ('soil carbon sequestration', 'health expenditure household burden', 0.05952278897166252),
    ('pesticide residue contamination', 'emergency department crowding', 0.05940064415335655),
    ('urban heat island effect', 'infectious disease contact tracing', 0.0591212622821331),
    ('maternal mortality risk factor', 'sea level rise projection', 0.05883873999118805),
    ('aquaculture fish farming', 'income inequality Gini coefficient', 0.058619577437639236),
    ('seed genetic modification', 'neighborhood segregation index', 0.058542121201753616),
    ('freshwater availability stress', 'mental health service access', 0.05847793444991112),
    ('public health surveillance system', 'sea level rise projection', 0.058435626327991486),
    ('food insecurity household level', 'ecosystem restoration practice', 0.05798289179801941),
    ('atmospheric carbon dioxide concentration', 'antibiotic resistance surveillance', 0.05757356807589531),
    ('plant disease surveillance', 'neighborhood segregation index', 0.05753999203443527),
    ('soil carbon sequestration', 'food insecurity household level', 0.057089898735284805),
    ('crop yield variability', 'labor market automation displacement', 0.05646149814128876),
    ('pollinator habitat loss', 'health insurance coverage gap', 0.055149368941783905),
    ('labor market automation displacement', 'mental health service access', 0.05475885793566704),
    ('soil carbon sequestration', 'racial discrimination employment', 0.054552074521780014),
    ('precision medicine genomic targeting', 'climate adaptation funding equity', 0.05356264486908913),
    ('plant disease surveillance', 'health equity racial disparity', 0.05351216718554497),
    ('food supply chain logistics', 'air quality particulate matter', 0.053299497812986374),
    ('public health surveillance system', 'racial discrimination employment', 0.05323256924748421),
    ('groundwater aquifer depletion', 'chronic disease self-management', 0.053008973598480225),
    ('seed genetic modification', 'racial discrimination employment', 0.05298658460378647),
    ('atmospheric carbon dioxide concentration', 'agroforestry land use', 0.052555765956640244),
    ('freshwater availability stress', 'neighborhood segregation index', 0.05214850977063179),
    ('infectious disease contact tracing', 'carbon emission trading', 0.05210527032613754),
    ('public health surveillance system', 'rural urban economic divide', 0.0520218163728714),
    ('pesticide residue contamination', 'precision medicine genomic targeting', 0.05183601751923561),
    ('biodiversity species extinction', 'precision medicine genomic targeting', 0.051818571984767914),
    ('aquaculture fish farming', 'disaster recovery economic resilience', 0.05168354511260986),
    ('soil carbon sequestration', 'maternal mortality risk factor', 0.051282722502946854),
    ('seed genetic modification', 'food insecurity household level', 0.05123760923743248),
    ('labor market automation displacement', 'ecosystem restoration practice', 0.05123011767864227),
    ('biodiversity species extinction', 'rural smallholder income', 0.05116784945130348),
    ('chronic disease self-management', 'freshwater availability stress', 0.0509725846350193),
    ('crop yield variability', 'neighborhood segregation index', 0.05046112835407257),
    ('pesticide residue contamination', 'disaster recovery economic resilience', 0.0503915511071682),
    ('food insecurity household level', 'infectious disease contact tracing', 0.04948806017637253),
    ('agricultural water use', 'urban heat island effect', 0.049461204558610916),
    ('carbon emission trading', 'health expenditure household burden', 0.04920836165547371),
    ('atmospheric carbon dioxide concentration', 'racial discrimination employment', 0.04919232800602913),
    ('deforestation tropical forest', 'social determinants of health', 0.04881877824664116),
    ('farm subsidy policy', 'sea level rise projection', 0.048653800040483475),
    ('chemical contaminant bioaccumulation', 'pediatric nutrition intervention', 0.048246145248413086),
    ('pediatric nutrition intervention', 'renewable energy transition', 0.047244489192962646),
    ('chemical contaminant bioaccumulation', 'social determinants of health', 0.046953052282333374),
    ('biodiversity species extinction', 'health equity racial disparity', 0.0469370111823082),
    ('income inequality Gini coefficient', 'ecosystem restoration practice', 0.046483129262924194),
    ('groundwater aquifer depletion', 'aquaculture fish farming', 0.046205513179302216),
    ('carbon emission trading', 'social determinants of health', 0.04607276991009712),
    ('atmospheric carbon dioxide concentration', 'infectious disease contact tracing', 0.04568152874708176),
    ('pesticide residue contamination', 'neighborhood segregation index', 0.04556433483958244),
    ('public health surveillance system', 'renewable energy transition', 0.0453360341489315),
    ('freshwater availability stress', 'carbon tax distributional effect', 0.04515202343463898),
    ('groundwater aquifer depletion', 'income inequality Gini coefficient', 0.04502144828438759),
    ('antibiotic resistance surveillance', 'crop yield variability', 0.044674865901470184),
    ('freshwater availability stress', 'health insurance coverage gap', 0.04428698495030403),
    ('pollinator habitat loss', 'antibiotic resistance surveillance', 0.044096313416957855),
    ('freshwater availability stress', 'racial discrimination employment', 0.044045329093933105),
    ('food supply chain logistics', 'infectious disease contact tracing', 0.04377272352576256),
    ('rural urban economic divide', 'freshwater availability stress', 0.04332714155316353),
    ('groundwater aquifer depletion', 'neighborhood segregation index', 0.0431172177195549),
    ('plant disease surveillance', 'rural urban economic divide', 0.04303864389657974),
    ('carbon emission trading', 'neighborhood segregation index', 0.042867906391620636),
    ('groundwater aquifer depletion', 'agricultural labor migration', 0.042831577360630035),
    ('aquaculture fish farming', 'neighborhood segregation index', 0.04247044399380684),
    ('precision medicine genomic targeting', 'sea level rise projection', 0.042323462665081024),
    ('chemical contaminant bioaccumulation', 'precision medicine genomic targeting', 0.0422661192715168),
    ('chemical contaminant bioaccumulation', 'rural smallholder income', 0.04125062748789787),
    ('food insecurity household level', 'sea level rise projection', 0.04104683920741081),
    ('agricultural water use', 'social determinants of health', 0.04072902724146843),
    ('social determinants of health', 'unemployment benefit duration', 0.04062337428331375),
    ('chemical contaminant bioaccumulation', 'rural urban economic divide', 0.040310442447662354),
    ('precision medicine genomic targeting', 'racial discrimination employment', 0.040002983063459396),
    ('pediatric nutrition intervention', 'income inequality Gini coefficient', 0.039917804300785065),
    ('antibiotic resistance surveillance', 'neighborhood segregation index', 0.03975723311305046),
    ('renewable energy transition', 'mental health service access', 0.03964078426361084),
    ('biodiversity species extinction', 'rural urban economic divide', 0.03940143063664436),
    ('atmospheric carbon dioxide concentration', 'disaster recovery economic resilience', 0.039380840957164764),
    ('labor market automation displacement', 'social determinants of health', 0.039338719099760056),
    ('air quality particulate matter', 'racial discrimination employment', 0.03925410658121109),
    ('income inequality Gini coefficient', 'freshwater availability stress', 0.03907936066389084),
    ('carbon tax distributional effect', 'health equity racial disparity', 0.03878995403647423),
    ('air quality particulate matter', 'labor market automation displacement', 0.0387413464486599),
    ('aquaculture fish farming', 'health insurance coverage gap', 0.03854743391275406),
    ('farm subsidy policy', 'urban heat island effect', 0.03847337141633034),
    ('rural urban economic divide', 'mental health service access', 0.03791951760649681),
    ('pollinator habitat loss', 'sea level rise projection', 0.03785840421915054),
    ('groundwater aquifer depletion', 'racial discrimination employment', 0.037555381655693054),
    ('agroforestry land use', 'unemployment benefit duration', 0.037369504570961),
    ('soil carbon sequestration', 'urban heat island effect', 0.03727802634239197),
    ('biodiversity species extinction', 'health insurance coverage gap', 0.03718782216310501),
    ('pesticide residue contamination', 'drought frequency projection', 0.03713848069310188),
    ('plant disease surveillance', 'carbon emission trading', 0.036914482712745667),
    ('seed genetic modification', 'health insurance coverage gap', 0.03687245771288872),
    ('crop yield variability', 'urban heat island effect', 0.036550555378198624),
    ('aquaculture fish farming', 'maternal mortality risk factor', 0.036346565932035446),
    ('chemical contaminant bioaccumulation', 'health expenditure household burden', 0.03619098290801048),
    ('seed genetic modification', 'health equity racial disparity', 0.035424716770648956),
    ('poverty alleviation microfinance', 'sea level rise projection', 0.03509272262454033),
    ('crop yield variability', 'health insurance coverage gap', 0.03507831692695618),
    ('aquaculture fish farming', 'emergency department crowding', 0.03503244370222092),
    ('deforestation tropical forest', 'antibiotic resistance surveillance', 0.034988295286893845),
    ('pediatric nutrition intervention', 'sea level rise projection', 0.03446100652217865),
    ('antibiotic resistance surveillance', 'agricultural labor migration', 0.03445930406451225),
    ('deforestation tropical forest', 'neighborhood segregation index', 0.03425448387861252),
    ('air quality particulate matter', 'chronic disease self-management', 0.03406316414475441),
    ('freshwater availability stress', 'infectious disease contact tracing', 0.03394165635108948),
    ('precision medicine genomic targeting', 'drought frequency projection', 0.03383076190948486),
    ('public health surveillance system', 'ecosystem restoration practice', 0.03366437181830406),
    ('antibiotic resistance surveillance', 'food supply chain logistics', 0.03364093229174614),
    ('soil carbon sequestration', 'health equity racial disparity', 0.0334932915866375),
    ('biodiversity species extinction', 'farm subsidy policy', 0.033131711184978485),
    ('pollinator habitat loss', 'racial discrimination employment', 0.03308302164077759),
    ('precision medicine genomic targeting', 'air quality particulate matter', 0.03300889581441879),
    ('deforestation tropical forest', 'labor market automation displacement', 0.03300202265381813),
    ('groundwater aquifer depletion', 'carbon tax distributional effect', 0.032638005912303925),
    ('public health surveillance system', 'crop yield variability', 0.032188672572374344),
    ('agricultural water use', 'racial discrimination employment', 0.03173353523015976),
    ('pollinator habitat loss', 'public health surveillance system', 0.031494345515966415),
    ('aquaculture fish farming', 'carbon tax distributional effect', 0.031088685616850853),
    ('plant disease surveillance', 'racial discrimination employment', 0.03091171197593212),
    ('urban heat island effect', 'maternal mortality risk factor', 0.030823571607470512),
    ('drought frequency projection', 'neighborhood segregation index', 0.03059554658830166),
    ('pesticide residue contamination', 'mental health service access', 0.030490316450595856),
    ('seed genetic modification', 'rural urban economic divide', 0.030415911227464676),
    ('agroforestry land use', 'health insurance coverage gap', 0.03033282421529293),
    ('atmospheric carbon dioxide concentration', 'farm subsidy policy', 0.029997702687978745),
    ('chemical contaminant bioaccumulation', 'mental health service access', 0.02923438511788845),
    ('public health surveillance system', 'rural smallholder income', 0.029072336852550507),
    ('chronic disease self-management', 'carbon emission trading', 0.02888154610991478),
    ('chemical contaminant bioaccumulation', 'poverty alleviation microfinance', 0.027480553835630417),
    ('precision medicine genomic targeting', 'crop yield variability', 0.0274395439773798),
    ('atmospheric carbon dioxide concentration', 'health insurance coverage gap', 0.02743474580347538),
    ('public health surveillance system', 'seed genetic modification', 0.02726334147155285),
    ('atmospheric carbon dioxide concentration', 'plant disease surveillance', 0.02711549773812294),
    ('drought frequency projection', 'maternal mortality risk factor', 0.026977261528372765),
    ('public health surveillance system', 'carbon emission trading', 0.02678842470049858),
    ('atmospheric carbon dioxide concentration', 'mental health service access', 0.02637910656630993),
    ('chronic disease self-management', 'renewable energy transition', 0.026088567450642586),
    ('plant disease surveillance', 'income inequality Gini coefficient', 0.026014234870672226),
    ('pesticide residue contamination', 'labor market automation displacement', 0.025363007560372353),
    ('pollinator habitat loss', 'unemployment benefit duration', 0.025272315368056297),
    ('precision medicine genomic targeting', 'health expenditure household burden', 0.025076214224100113),
    ('seed genetic modification', 'disaster recovery economic resilience', 0.025054942816495895),
    ('biodiversity species extinction', 'income inequality Gini coefficient', 0.024323400110006332),
    ('plant disease surveillance', 'climate adaptation funding equity', 0.0242796428501606),
    ('atmospheric carbon dioxide concentration', 'food supply chain logistics', 0.024131156504154205),
    ('poverty alleviation microfinance', 'infectious disease contact tracing', 0.024121146649122238),
    ('income inequality Gini coefficient', 'air quality particulate matter', 0.024001024663448334),
    ('atmospheric carbon dioxide concentration', 'health equity racial disparity', 0.02360965684056282),
    ('pesticide residue contamination', 'deforestation tropical forest', 0.023570582270622253),
    ('rural smallholder income', 'chronic disease self-management', 0.02342471107840538),
    ('soil carbon sequestration', 'chronic disease self-management', 0.023421848192811012),
    ('agricultural water use', 'unemployment benefit duration', 0.023389847949147224),
    ('seed genetic modification', 'carbon tax distributional effect', 0.02263922244310379),
    ('groundwater aquifer depletion', 'health expenditure household burden', 0.022486118599772453),
    ('chronic disease self-management', 'labor market automation displacement', 0.022417904809117317),
    ('pesticide residue contamination', 'poverty alleviation microfinance', 0.021917007863521576),
    ('agricultural water use', 'neighborhood segregation index', 0.021915340796113014),
    ('crop yield variability', 'racial discrimination employment', 0.021354513242840767),
    ('chemical contaminant bioaccumulation', 'racial discrimination employment', 0.02132357470691204),
    ('soil carbon sequestration', 'neighborhood segregation index', 0.020900506526231766),
    ('precision medicine genomic targeting', 'rural smallholder income', 0.0206894651055336),
    ('maternal mortality risk factor', 'racial discrimination employment', 0.02050967514514923),
    ('precision medicine genomic targeting', 'agricultural labor migration', 0.020262975245714188),
    ('pollinator habitat loss', 'social determinants of health', 0.02025083266198635),
    ('emergency department crowding', 'drought frequency projection', 0.019509954378008842),
    ('crop yield variability', 'emergency department crowding', 0.019367586821317673),
    ('food supply chain logistics', 'social determinants of health', 0.01914069801568985),
    ('air quality particulate matter', 'mental health service access', 0.01911792904138565),
    ('agricultural water use', 'health equity racial disparity', 0.019056927412748337),
    ('biodiversity species extinction', 'social determinants of health', 0.019024169072508812),
    ('atmospheric carbon dioxide concentration', 'neighborhood segregation index', 0.01829446852207184),
    ('chemical contaminant bioaccumulation', 'neighborhood segregation index', 0.017658120021224022),
    ('plant disease surveillance', 'urban heat island effect', 0.017299188300967216),
    ('rural smallholder income', 'air quality particulate matter', 0.01725694350898266),
    ('public health surveillance system', 'poverty alleviation microfinance', 0.017221054062247276),
    ('aquaculture fish farming', 'infectious disease contact tracing', 0.017026545479893684),
    ('soil carbon sequestration', 'public health surveillance system', 0.016852019354701042),
    ('pediatric nutrition intervention', 'agricultural labor migration', 0.016518911346793175),
    ('chemical contaminant bioaccumulation', 'chronic disease self-management', 0.016133833676576614),
    ('agricultural water use', 'precision medicine genomic targeting', 0.01571611873805523),
    ('income inequality Gini coefficient', 'urban heat island effect', 0.01569412276148796),
    ('pesticide residue contamination', 'racial discrimination employment', 0.015154046006500721),
    ('pollinator habitat loss', 'pediatric nutrition intervention', 0.014494435861706734),
    ('pediatric nutrition intervention', 'unemployment benefit duration', 0.01408820878714323),
    ('plant disease surveillance', 'poverty alleviation microfinance', 0.01399734616279602),
    ('agricultural water use', 'chronic disease self-management', 0.013819515705108643),
    ('food supply chain logistics', 'health insurance coverage gap', 0.013812742196023464),
    ('pollinator habitat loss', 'chronic disease self-management', 0.013748091645538807),
    ('carbon emission trading', 'health insurance coverage gap', 0.013736382126808167),
    ('infectious disease contact tracing', 'disaster recovery economic resilience', 0.013703293167054653),
    ('agricultural labor migration', 'sea level rise projection', 0.012827926315367222),
    ('aquaculture fish farming', 'mental health service access', 0.012542768381536007),
    ('urban heat island effect', 'mental health service access', 0.012538230046629906),
    ('biodiversity species extinction', 'health expenditure household burden', 0.012309243902564049),
    ('pesticide residue contamination', 'pediatric nutrition intervention', 0.01230782549828291),
    ('agricultural labor migration', 'air quality particulate matter', 0.010962015017867088),
    ('agricultural water use', 'antibiotic resistance surveillance', 0.010740583762526512),
    ('atmospheric carbon dioxide concentration', 'aquaculture fish farming', 0.010692675597965717),
    ('chronic disease self-management', 'racial discrimination employment', 0.01062525250017643),
    ('food insecurity household level', 'renewable energy transition', 0.010199062526226044),
    ('groundwater aquifer depletion', 'health insurance coverage gap', 0.010097909718751907),
    ('emergency department crowding', 'unemployment benefit duration', 0.009963219054043293),
    ('biodiversity species extinction', 'food supply chain logistics', 0.009789674542844296),
    ('food supply chain logistics', 'health equity racial disparity', 0.009191213175654411),
    ('groundwater aquifer depletion', 'mental health service access', 0.008232833817601204),
    ('food supply chain logistics', 'precision medicine genomic targeting', 0.008189854212105274),
    ('soil carbon sequestration', 'infectious disease contact tracing', 0.00794366654008627),
    ('food insecurity household level', 'drought frequency projection', 0.007633234839886427),
    ('antibiotic resistance surveillance', 'ecosystem restoration practice', 0.007617445662617683),
    ('pediatric nutrition intervention', 'drought frequency projection', 0.007490651216357946),
    ('food supply chain logistics', 'neighborhood segregation index', 0.007462900131940842),
    ('agricultural water use', 'health insurance coverage gap', 0.006960235070437193),
    ('seed genetic modification', 'carbon emission trading', 0.006524672266095877),
    ('air quality particulate matter', 'disaster recovery economic resilience', 0.006171099375933409),
    ('chemical contaminant bioaccumulation', 'health equity racial disparity', 0.005994101986289024),
    ('maternal mortality risk factor', 'ecosystem restoration practice', 0.005975354462862015),
    ('infectious disease contact tracing', 'unemployment benefit duration', 0.005709605757147074),
    ('groundwater aquifer depletion', 'pediatric nutrition intervention', 0.0055439225398004055),
    ('chemical contaminant bioaccumulation', 'labor market automation displacement', 0.0053936513140797615),
    ('antibiotic resistance surveillance', 'air quality particulate matter', 0.0052574104629457),
    ('crop yield variability', 'air quality particulate matter', 0.005254713818430901),
    ('maternal mortality risk factor', 'labor market automation displacement', 0.005151168908923864),
    ('infectious disease contact tracing', 'ecosystem restoration practice', 0.0051505304872989655),
    ('public health surveillance system', 'carbon tax distributional effect', 0.0050562466494739056),
    ('rural smallholder income', 'unemployment benefit duration', 0.004952605348080397),
    ('groundwater aquifer depletion', 'infectious disease contact tracing', 0.0037840420845896006),
    ('plant disease surveillance', 'carbon tax distributional effect', 0.0033467188477516174),
    ('precision medicine genomic targeting', 'neighborhood segregation index', 0.003321688622236252),
    ('crop yield variability', 'social determinants of health', 0.003133809193968773),
    ('atmospheric carbon dioxide concentration', 'pediatric nutrition intervention', 0.0018961665919050574),
    ('sea level rise projection', 'health insurance coverage gap', 0.001536073163151741),
    ('precision medicine genomic targeting', 'carbon emission trading', 0.0014064532006159425),
    ('carbon emission trading', 'health equity racial disparity', 0.001268421416170895),
    ('soil carbon sequestration', 'income inequality Gini coefficient', 0.0011507157469168305),
    ('atmospheric carbon dioxide concentration', 'rural urban economic divide', 0.000760179478675127),
    ('precision medicine genomic targeting', 'farm subsidy policy', 0.000646820873953402),
    ('sea level rise projection', 'neighborhood segregation index', 0.0006064624176360667),
    ('rural smallholder income', 'infectious disease contact tracing', 0.00025023968191817403),
    ('pesticide residue contamination', 'social determinants of health', -0.0007316645351238549),
    ('pediatric nutrition intervention', 'neighborhood segregation index', -0.001059787580743432),
    ('renewable energy transition', 'social determinants of health', -0.0017188802594318986),
    ('agroforestry land use', 'precision medicine genomic targeting', -0.001875875168479979),
    ('crop yield variability', 'health equity racial disparity', -0.0018864409066736698),
    ('agricultural labor migration', 'mental health service access', -0.0027407058514654636),
    ('antibiotic resistance surveillance', 'racial discrimination employment', -0.0028266324661672115),
    ('emergency department crowding', 'sea level rise projection', -0.004119870252907276),
    ('ecosystem restoration practice', 'health insurance coverage gap', -0.004289615899324417),
    ('chronic disease self-management', 'carbon tax distributional effect', -0.004438947886228561),
    ('income inequality Gini coefficient', 'chronic disease self-management', -0.004507383331656456),
    ('soil carbon sequestration', 'antibiotic resistance surveillance', -0.00462600402534008),
    ('pesticide residue contamination', 'sea level rise projection', -0.00524474261328578),
    ('rural urban economic divide', 'sea level rise projection', -0.005745489615947008),
    ('biodiversity species extinction', 'food insecurity household level', -0.005924230441451073),
    ('groundwater aquifer depletion', 'food supply chain logistics', -0.006101023871451616),
    ('seed genetic modification', 'air quality particulate matter', -0.006362191867083311),
    ('atmospheric carbon dioxide concentration', 'rural smallholder income', -0.007215515710413456),
    ('public health surveillance system', 'freshwater availability stress', -0.007466751150786877),
    ('deforestation tropical forest', 'health equity racial disparity', -0.008335850201547146),
    ('atmospheric carbon dioxide concentration', 'food insecurity household level', -0.008587337099015713),
    ('pediatric nutrition intervention', 'urban heat island effect', -0.008823787793517113),
    ('public health surveillance system', 'climate adaptation funding equity', -0.008827002719044685),
    ('pollinator habitat loss', 'precision medicine genomic targeting', -0.008827191777527332),
    ('biodiversity species extinction', 'unemployment benefit duration', -0.008880048990249634),
    ('antibiotic resistance surveillance', 'freshwater availability stress', -0.00907269399613142),
    ('plant disease surveillance', 'sea level rise projection', -0.009141946211457253),
    ('public health surveillance system', 'agricultural labor migration', -0.009683037176728249),
    ('aquaculture fish farming', 'unemployment benefit duration', -0.010091024450957775),
    ('crop yield variability', 'chronic disease self-management', -0.010449654422700405),
    ('rural urban economic divide', 'drought frequency projection', -0.010737843811511993),
    ('sea level rise projection', 'health equity racial disparity', -0.010995126329362392),
    ('antibiotic resistance surveillance', 'labor market automation displacement', -0.011066166684031487),
    ('biodiversity species extinction', 'labor market automation displacement', -0.012057976797223091),
    ('pesticide residue contamination', 'urban heat island effect', -0.012604947201907635),
    ('renewable energy transition', 'neighborhood segregation index', -0.012958146631717682),
    ('rural urban economic divide', 'chronic disease self-management', -0.01326688937842846),
    ('pediatric nutrition intervention', 'carbon tax distributional effect', -0.013686489313840866),
    ('crop yield variability', 'infectious disease contact tracing', -0.013783574104309082),
    ('seed genetic modification', 'chronic disease self-management', -0.0139307901263237),
    ('chemical contaminant bioaccumulation', 'farm subsidy policy', -0.014222878962755203),
    ('seed genetic modification', 'unemployment benefit duration', -0.014271115884184837),
    ('food supply chain logistics', 'maternal mortality risk factor', -0.01436374057084322),
    ('seed genetic modification', 'urban heat island effect', -0.014509729109704494),
    ('carbon emission trading', 'unemployment benefit duration', -0.014560096897184849),
    ('sea level rise projection', 'social determinants of health', -0.014621159993112087),
    ('chemical contaminant bioaccumulation', 'climate adaptation funding equity', -0.014857579953968525),
    ('deforestation tropical forest', 'health insurance coverage gap', -0.015277289785444736),
    ('plant disease surveillance', 'disaster recovery economic resilience', -0.01552631240338087),
    ('renewable energy transition', 'racial discrimination employment', -0.015800854191184044),
    ('seed genetic modification', 'mental health service access', -0.016014929860830307),
    ('aquaculture fish farming', 'air quality particulate matter', -0.01605992205440998),
    ('drought frequency projection', 'health insurance coverage gap', -0.0161532461643219),
    ('emergency department crowding', 'renewable energy transition', -0.016413412988185883),
    ('groundwater aquifer depletion', 'plant disease surveillance', -0.01678660698235035),
    ('pollinator habitat loss', 'mental health service access', -0.01683781109750271),
    ('seed genetic modification', 'infectious disease contact tracing', -0.016929712146520615),
    ('poverty alleviation microfinance', 'air quality particulate matter', -0.016960114240646362),
    ('atmospheric carbon dioxide concentration', 'poverty alleviation microfinance', -0.017226848751306534),
    ('aquaculture fish farming', 'urban heat island effect', -0.01724928617477417),
    ('ecosystem restoration practice', 'mental health service access', -0.0182256530970335),
    ('infectious disease contact tracing', 'racial discrimination employment', -0.018669798970222473),
    ('groundwater aquifer depletion', 'antibiotic resistance surveillance', -0.019268622621893883),
    ('agroforestry land use', 'social determinants of health', -0.01945737563073635),
    ('groundwater aquifer depletion', 'food insecurity household level', -0.019474472850561142),
    ('pesticide residue contamination', 'chronic disease self-management', -0.019610010087490082),
    ('seed genetic modification', 'freshwater availability stress', -0.01994035579264164),
    ('pediatric nutrition intervention', 'carbon emission trading', -0.01997753418982029),
    ('rural smallholder income', 'sea level rise projection', -0.020122194662690163),
    ('chronic disease self-management', 'unemployment benefit duration', -0.020199574530124664),
    ('chemical contaminant bioaccumulation', 'income inequality Gini coefficient', -0.020696695894002914),
    ('ecosystem restoration practice', 'health expenditure household burden', -0.02111540548503399),
    ('public health surveillance system', 'labor market automation displacement', -0.02186228148639202),
    ('antibiotic resistance surveillance', 'sea level rise projection', -0.022165384143590927),
    ('food supply chain logistics', 'urban heat island effect', -0.02243860997259617),
    ('public health surveillance system', 'unemployment benefit duration', -0.02290498837828636),
    ('pesticide residue contamination', 'unemployment benefit duration', -0.022951764985919),
    ('seed genetic modification', 'emergency department crowding', -0.02426176518201828),
    ('groundwater aquifer depletion', 'health equity racial disparity', -0.024918075650930405),
    ('maternal mortality risk factor', 'carbon emission trading', -0.02564644254744053),
    ('rural urban economic divide', 'infectious disease contact tracing', -0.025849124416708946),
    ('atmospheric carbon dioxide concentration', 'labor market automation displacement', -0.026785647496581078),
    ('pediatric nutrition intervention', 'racial discrimination employment', -0.02680141106247902),
    ('emergency department crowding', 'carbon emission trading', -0.026864780113101006),
    ('deforestation tropical forest', 'food insecurity household level', -0.026866290718317032),
    ('soil carbon sequestration', 'mental health service access', -0.027839038521051407),
    ('antibiotic resistance surveillance', 'carbon emission trading', -0.02926172874867916),
    ('carbon emission trading', 'mental health service access', -0.03009827993810177),
    ('agroforestry land use', 'antibiotic resistance surveillance', -0.030368294566869736),
    ('groundwater aquifer depletion', 'seed genetic modification', -0.030801165848970413),
    ('atmospheric carbon dioxide concentration', 'unemployment benefit duration', -0.031400907784700394),
    ('precision medicine genomic targeting', 'unemployment benefit duration', -0.031654804944992065),
    ('precision medicine genomic targeting', 'labor market automation displacement', -0.031890831887722015),
    ('biodiversity species extinction', 'pediatric nutrition intervention', -0.032201431691646576),
    ('plant disease surveillance', 'unemployment benefit duration', -0.03381416201591492),
    ('chronic disease self-management', 'urban heat island effect', -0.034068647772073746),
    ('antibiotic resistance surveillance', 'poverty alleviation microfinance', -0.03409108892083168),
    ('ecosystem restoration practice', 'social determinants of health', -0.03449391573667526),
    ('precision medicine genomic targeting', 'ecosystem restoration practice', -0.035201165825128555),
    ('precision medicine genomic targeting', 'income inequality Gini coefficient', -0.03564058244228363),
    ('pesticide residue contamination', 'renewable energy transition', -0.036390725523233414),
    ('precision medicine genomic targeting', 'rural urban economic divide', -0.03770049288868904),
    ('income inequality Gini coefficient', 'emergency department crowding', -0.037899356335401535),
    ('crop yield variability', 'mental health service access', -0.03799242153763771),
    ('antibiotic resistance surveillance', 'climate adaptation funding equity', -0.038854438811540604),
    ('soil carbon sequestration', 'precision medicine genomic targeting', -0.038941510021686554),
    ('soil carbon sequestration', 'unemployment benefit duration', -0.03958911448717117),
    ('pollinator habitat loss', 'health equity racial disparity', -0.04083075374364853),
    ('seed genetic modification', 'social determinants of health', -0.04099409282207489),
    ('antibiotic resistance surveillance', 'unemployment benefit duration', -0.04107988253235817),
    ('precision medicine genomic targeting', 'renewable energy transition', -0.04125578701496124),
    ('groundwater aquifer depletion', 'social determinants of health', -0.04242650419473648),
    ('deforestation tropical forest', 'unemployment benefit duration', -0.042492933571338654),
    ('precision medicine genomic targeting', 'freshwater availability stress', -0.043015748262405396),
    ('deforestation tropical forest', 'pediatric nutrition intervention', -0.04356043040752411),
    ('atmospheric carbon dioxide concentration', 'agricultural labor migration', -0.04361334815621376),
    ('chronic disease self-management', 'neighborhood segregation index', -0.04402296245098114),
    ('infectious disease contact tracing', 'carbon tax distributional effect', -0.04417865723371506),
    ('seed genetic modification', 'labor market automation displacement', -0.044858064502477646),
    ('plant disease surveillance', 'labor market automation displacement', -0.04488303139805794),
    ('groundwater aquifer depletion', 'precision medicine genomic targeting', -0.04547302797436714),
    ('pediatric nutrition intervention', 'labor market automation displacement', -0.046197958290576935),
    ('farm subsidy policy', 'air quality particulate matter', -0.046249870210886),
    ('pesticide residue contamination', 'climate adaptation funding equity', -0.04630785062909126),
    ('antibiotic resistance surveillance', 'drought frequency projection', -0.046556390821933746),
    ('atmospheric carbon dioxide concentration', 'seed genetic modification', -0.04667385295033455),
    ('seed genetic modification', 'sea level rise projection', -0.04696614295244217),
    ('public health surveillance system', 'income inequality Gini coefficient', -0.047606635838747025),
    ('agricultural labor migration', 'chronic disease self-management', -0.047772396355867386),
    ('groundwater aquifer depletion', 'public health surveillance system', -0.0478733628988266),
    ('antibiotic resistance surveillance', 'rural smallholder income', -0.04806622490286827),
    ('deforestation tropical forest', 'mental health service access', -0.04875144734978676),
    ('air quality particulate matter', 'unemployment benefit duration', -0.05077100545167923),
    ('chemical contaminant bioaccumulation', 'unemployment benefit duration', -0.05437181890010834),
    ('seed genetic modification', 'health expenditure household burden', -0.05449718236923218),
    ('renewable energy transition', 'health equity racial disparity', -0.054926518350839615),
    ('antibiotic resistance surveillance', 'rural urban economic divide', -0.055131696164608),
    ('soil carbon sequestration', 'health insurance coverage gap', -0.056072335690259933),
    ('ecosystem restoration practice', 'unemployment benefit duration', -0.05754335597157478),
    ('emergency department crowding', 'carbon tax distributional effect', -0.057951003313064575),
    ('pesticide residue contamination', 'health equity racial disparity', -0.05829673632979393),
    ('drought frequency projection', 'social determinants of health', -0.05924458056688309),
    ('food supply chain logistics', 'unemployment benefit duration', -0.05971499904990196),
    ('income inequality Gini coefficient', 'infectious disease contact tracing', -0.06134778633713722),
    ('farm subsidy policy', 'infectious disease contact tracing', -0.06185799837112427),
    ('infectious disease contact tracing', 'renewable energy transition', -0.06351566314697266),
    ('food supply chain logistics', 'income inequality Gini coefficient', -0.06438682973384857),
    ('deforestation tropical forest', 'precision medicine genomic targeting', -0.06510286033153534),
    ('maternal mortality risk factor', 'renewable energy transition', -0.06716125458478928),
    ('antibiotic resistance surveillance', 'urban heat island effect', -0.0680326446890831),
    ('precision medicine genomic targeting', 'urban heat island effect', -0.07135731726884842),
    ('soil carbon sequestration', 'social determinants of health', -0.07154477387666702),
    ('antibiotic resistance surveillance', 'disaster recovery economic resilience', -0.07209707051515579),
    ('antibiotic resistance surveillance', 'renewable energy transition', -0.07492812722921371),
    ('pediatric nutrition intervention', 'air quality particulate matter', -0.07896903902292252),
    ('biodiversity species extinction', 'mental health service access', -0.08036398142576218),
    ('carbon tax distributional effect', 'mental health service access', -0.08156345039606094),
    ('pesticide residue contamination', 'health insurance coverage gap', -0.08312848955392838),
    ('drought frequency projection', 'health equity racial disparity', -0.08456631749868393),
    ('precision medicine genomic targeting', 'disaster recovery economic resilience', -0.0853208526968956),
    ('atmospheric carbon dioxide concentration', 'precision medicine genomic targeting', -0.08712638169527054),
    ('precision medicine genomic targeting', 'food insecurity household level', -0.08766110986471176),
    ('pesticide residue contamination', 'income inequality Gini coefficient', -0.08782145380973816),
    ('infectious disease contact tracing', 'climate adaptation funding equity', -0.09142718464136124),
    ('precision medicine genomic targeting', 'carbon tax distributional effect', -0.09613595902919769),
    ('chronic disease self-management', 'sea level rise projection', -0.09671909362077713),
    ('antibiotic resistance surveillance', 'income inequality Gini coefficient', -0.10672511905431747),
    ('antibiotic resistance surveillance', 'carbon tax distributional effect', -0.11945926398038864),
    ('chemical contaminant bioaccumulation', 'health insurance coverage gap', -0.1234954223036766),
]


def build_scaffold_pairs():
    scored = []
    for a, b in combinations(NAMES, 2):
        if DOMAIN[a] == DOMAIN[b]:
            continue
        scored.append((a, b, scaffold_score(a, b)))
    return scored


def build_minilm_pairs():
    return [(a, b, s) for a, b, s in EMBEDDING_PAIRS]


if __name__ == "__main__":
    print("SCAFFOLD vs MiniLM — same 25-pair gold, cross-domain only")
    print("Scaffold = theme lexicon + token Jaccard on concept labels.")
    print("Gold is not used to generate or rank predictions.\n")

    scaf = build_scaffold_pairs()
    mini = build_minilm_pairs()
    print(f"Scaffold candidates: {len(scaf)}")
    print(f"MiniLM candidates:   {len(mini)}")
    print(f"Gold positives:      {len(GOLD)}")

    scaf_ap = ap_score(scaf, GOLD_LIST)
    mini_ap = ap_score(mini, GOLD_LIST)
    print(f"\nRanking AP  scaffold={scaf_ap:.3f}  MiniLM={mini_ap:.3f}")

    taus = [0.10, 0.20, 0.25, 0.30, 0.35, 0.40, 0.50, 0.60, 0.70]
    best_s = print_sweep("TABLE S — Ontological scaffold", scaf, GOLD_LIST, taus)
    best_m = print_sweep("TABLE B — MiniLM all-MiniLM-L6-v2", mini, GOLD_LIST, [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9])

    # head-to-head at MiniLM default τ=0.5 and each method's peak F1
    s05 = evaluate_threshold(scaf, 0.50, GOLD_LIST)
    m05 = evaluate_threshold(mini, 0.50, GOLD_LIST)
    s_peak_t, s_peak, s_peak_n = best_s
    m_peak_t, m_peak, m_peak_n = best_m

    print("\n" + "=" * 88)
    print("TABLE 4 — Head-to-head on gold_relationships.json (25 pairs)")
    print("=" * 88)
    cols = [
        ("Scaffold τ=0.50", s05, sum(1 for *_, c in scaf if c >= 0.50)),
        (f"Scaffold peak τ={s_peak_t:.2f}", s_peak, s_peak_n),
        ("MiniLM τ=0.50", m05, sum(1 for *_, c in mini if c >= 0.50)),
        (f"MiniLM peak τ={m_peak_t:.2f}", m_peak, m_peak_n),
    ]
    print(f"{'Metric':<28}" + "".join(f"{c[0]:>18}" for c in cols))
    print("-" * 88)
    print(f"{'Gold (TP+FN)':<28}" + "".join(f"{25:18d}" for _ in cols))
    print(f"{'Predicted':<28}" + "".join(f"{c[2]:18d}" for c in cols))
    print(f"{'TP / FP / FN':<28}" + "".join(f"{c[1].tp}/{c[1].fp}/{c[1].fn}".rjust(18) for c in cols))
    print(f"{'Precision':<28}" + "".join(f"{c[1].precision:18.3f}" for c in cols))
    print(f"{'Recall':<28}" + "".join(f"{c[1].recall:18.3f}" for c in cols))
    print(f"{'F1':<28}" + "".join(f"{c[1].f1:18.3f}" for c in cols))
    print(f"{'FDR':<28}" + "".join(f"{c[1].fdr:18.3f}" for c in cols))
    print(f"{'AP (full ranking)':<28}{scaf_ap:18.3f}{scaf_ap:18.3f}{mini_ap:18.3f}{mini_ap:18.3f}")
    print()
    if s05.f1 > m05.f1:
        print(f"At τ=0.50 the scaffold F1 ({s05.f1:.3f}) exceeds MiniLM ({m05.f1:.3f}).")
    else:
        print(f"At τ=0.50 the scaffold F1 ({s05.f1:.3f}) does not exceed MiniLM ({m05.f1:.3f}).")
    if s_peak.f1 > m_peak.f1:
        print(f"Peak-F1: scaffold {s_peak.f1:.3f} @ τ={s_peak_t:.2f} exceeds MiniLM {m_peak.f1:.3f} @ τ={m_peak_t:.2f}.")
    else:
        print(f"Peak-F1: scaffold {s_peak.f1:.3f} @ τ={s_peak_t:.2f} does not exceed MiniLM {m_peak.f1:.3f} @ τ={m_peak_t:.2f}.")

    print("\nTop 12 scaffold predictions:")
    for a, b, c in sorted(scaf, key=lambda r: -r[2])[:12]:
        mark = "TP" if tuple(sorted((a, b))) in GOLD else "FP"
        print(f"  {c:.3f} {mark:2s}  {a}  <->  {b}")

    OUT.mkdir(exist_ok=True)
    with (OUT / "scaffold_vs_minilm.csv").open("w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["system", "tau", "predicted", "tp", "fp", "fn", "precision", "recall", "f1", "fdr", "ap"])
        w.writerow(["scaffold", 0.50, sum(1 for *_, c in scaf if c >= 0.50), s05.tp, s05.fp, s05.fn, s05.precision, s05.recall, s05.f1, s05.fdr, scaf_ap])
        w.writerow(["scaffold_peak", s_peak_t, s_peak_n, s_peak.tp, s_peak.fp, s_peak.fn, s_peak.precision, s_peak.recall, s_peak.f1, s_peak.fdr, scaf_ap])
        w.writerow(["minilm", 0.50, sum(1 for *_, c in mini if c >= 0.50), m05.tp, m05.fp, m05.fn, m05.precision, m05.recall, m05.f1, m05.fdr, mini_ap])
        w.writerow(["minilm_peak", m_peak_t, m_peak_n, m_peak.tp, m_peak.fp, m_peak.fn, m_peak.precision, m_peak.recall, m_peak.f1, m_peak.fdr, mini_ap])
    print(f"\nSaved {OUT / 'scaffold_vs_minilm.csv'}")

